# Canonical Protocol Manual Validation

This notebook manually validates the Phase 1 canonical thesis protocol workflow through the real CLI path.

What this notebook tests:
- canonical protocol dry-run execution (`--all-suites --dry-run`)
- one real execution for the smallest official suite
- protocol-level artifact emission and structure
- run-level artifact emission
- protocol metadata presence inside run artifacts (`config.json`, `metrics.json`)

This notebook is for manual operator/developer validation, not automated test coverage.

## Setup

In [3]:
import json
import os
import platform
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    cursor = (start or Path.cwd()).resolve()
    for candidate in (cursor, *cursor.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'Thesis_ML').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root (expected pyproject.toml and src/Thesis_ML).')

REPO_ROOT = find_repo_root()
PROTOCOL_PATH = REPO_ROOT / 'configs' / 'protocols' / 'thesis_canonical_v1.json'
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
MANUAL_VALIDATION_ROOT = REPO_ROOT / 'outputs' / 'manual_validation' / 'canonical_protocol'
DRY_RUN_REPORTS_ROOT = MANUAL_VALIDATION_ROOT / f'dry_run_{RUN_STAMP}'
REAL_RUN_REPORTS_ROOT = MANUAL_VALIDATION_ROOT / f'real_run_{RUN_STAMP}'

for path in (MANUAL_VALIDATION_ROOT, DRY_RUN_REPORTS_ROOT, REAL_RUN_REPORTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print('Repository root:', REPO_ROOT)
print('Protocol path:', PROTOCOL_PATH)
print('Dry-run reports root:', DRY_RUN_REPORTS_ROOT)
print('Real-run reports root:', REAL_RUN_REPORTS_ROOT)
print('Python executable:', sys.executable)
print('Python version:', platform.python_version())
print('Platform:', platform.platform())


Repository root: D:\Khaled\Projects\VScode\Thesis
Protocol path: D:\Khaled\Projects\VScode\Thesis\configs\protocols\thesis_canonical_v1.json
Dry-run reports root: D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\canonical_protocol\dry_run_20260315_020835
Real-run reports root: D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\canonical_protocol\real_run_20260315_020835
Python executable: d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe
Python version: 3.13.6
Platform: Windows-10-10.0.19045-SP0


## Preflight Checks and Helpers

In [4]:
assert PROTOCOL_PATH.exists(), f'Protocol file is missing: {PROTOCOL_PATH}'

protocol_payload = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
assert isinstance(protocol_payload, dict), 'Protocol JSON must be an object.'
assert protocol_payload.get('official_run_suites'), 'Protocol has no official_run_suites.'
print('Protocol ID:', protocol_payload.get('protocol_id'))
print('Protocol version:', protocol_payload.get('protocol_version'))
print('Protocol suites:', [suite['suite_id'] for suite in protocol_payload['official_run_suites']])

def env_with_repo_src() -> dict[str, str]:
    env = os.environ.copy()
    src_path = str(REPO_ROOT / 'src')
    existing = env.get('PYTHONPATH', '')
    env['PYTHONPATH'] = src_path if not existing else src_path + os.pathsep + existing
    return env

def _probe_command(command: list[str]) -> tuple[subprocess.CompletedProcess[str] | None, str | None]:
    try:
        completed = subprocess.run(
            command,
            cwd=str(REPO_ROOT),
            env=env_with_repo_src(),
            capture_output=True,
            text=True,
        )
        return completed, None
    except FileNotFoundError as exc:
        return None, f'command not found: {command[0]} ({exc})'
    except OSError as exc:
        return None, f'OS error while launching {command[0]}: {exc}'


def resolve_protocol_runner_base_command() -> list[str]:
    cli_cmd = ['thesisml-run-protocol']
    cli_probe, cli_launch_error = _probe_command(cli_cmd + ['--help'])
    if cli_probe is not None and cli_probe.returncode == 0:
        print('Using CLI entrypoint: thesisml-run-protocol')
        return cli_cmd

    module_cmd = [sys.executable, '-m', 'Thesis_ML.cli.protocol_runner']
    module_probe, module_launch_error = _probe_command(module_cmd + ['--help'])
    if module_probe is not None and module_probe.returncode == 0:
        print('Using module fallback: python -m Thesis_ML.cli.protocol_runner')
        return module_cmd

    cli_details = (
        cli_probe.stderr
        if cli_probe is not None
        else (cli_launch_error or 'unknown CLI launch error')
    )
    module_details = (
        module_probe.stderr
        if module_probe is not None
        else (module_launch_error or 'unknown module launch error')
    )

    raise RuntimeError(
        'Could not resolve protocol runner command via CLI or module fallback.\n'
        f'CLI details: {cli_details}\n'
        f'Module details: {module_details}'
    )


CLI_BASE_CMD = resolve_protocol_runner_base_command()

def run_subprocess(command: list[str], *, cwd: Path | None = None) -> dict[str, str | int]:
    resolved_cwd = cwd or REPO_ROOT
    print('$', ' '.join(shlex.quote(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=str(resolved_cwd),
        env=env_with_repo_src(),
        capture_output=True,
        text=True,
    )
    print('Return code:', completed.returncode)
    print('----- STDOUT -----')
    print(completed.stdout.strip() or '<empty>')
    print('----- STDERR -----')
    print(completed.stderr.strip() or '<empty>')
    return {'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr}

def parse_stdout_json(stdout: str) -> dict:
    text = stdout.strip()
    assert text, 'Expected JSON output on stdout but got empty output.'
    try:
        payload = json.loads(text)
    except json.JSONDecodeError as exc:
        raise AssertionError(f'Failed to parse CLI stdout as JSON: {exc}\nstdout:\n{text}') from exc
    assert isinstance(payload, dict), 'Expected JSON object on stdout.'
    return payload

def ensure_files_exist(base: Path, expected_files: list[str], *, label: str) -> None:
    missing = [str(base / name) for name in expected_files if not (base / name).exists()]
    assert not missing, f'{label}: missing expected files: {missing}'

def load_json(path: Path) -> dict:
    assert path.exists(), f'JSON file not found: {path}'
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert isinstance(payload, dict), f'Expected JSON object in {path}'
    return payload

def print_json_snippet(path: Path, *, max_chars: int = 2000) -> None:
    payload = load_json(path)
    text = json.dumps(payload, indent=2)
    if len(text) > max_chars:
        text = text[:max_chars] + '\n... [truncated] ...'
    print(text)

def print_tree(root: Path, *, max_depth: int = 3, max_entries: int = 200) -> None:
    root = root.resolve()
    print(root)
    if not root.exists():
        print('  <missing>')
        return
    count = 0
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        suffix = '/' if path.is_dir() else ''
        indent = '  ' * depth
        print(f'{indent}{rel}{suffix}')
        count += 1
        if count >= max_entries:
            print('... [tree output truncated] ...')
            break

EXPECTED_PROTOCOL_FILES = [
    'protocol.json',
    'compiled_protocol_manifest.json',
    'claim_to_run_map.json',
    'suite_summary.json',
    'execution_status.json',
    'report_index.csv',
]

EXPECTED_RUN_FILES = [
    'config.json',
    'metrics.json',
    'fold_metrics.csv',
    'fold_splits.csv',
    'predictions.csv',
    'spatial_compatibility_report.json',
    'interpretability_summary.json',
]


Protocol ID: thesis-canonical
Protocol version: 1.0.0
Protocol suites: ['primary_within_subject', 'secondary_cross_person_transfer', 'primary_controls']
Using module fallback: python -m Thesis_ML.cli.protocol_runner


## Dry-Run Canonical Protocol (`--all-suites`)

In [5]:
dry_run_cmd = CLI_BASE_CMD + [
    '--protocol', str(PROTOCOL_PATH),
    '--all-suites',
    '--reports-root', str(DRY_RUN_REPORTS_ROOT),
    '--dry-run',
]

dry_run_result = run_subprocess(dry_run_cmd)
assert dry_run_result['returncode'] == 0, 'Dry-run command failed.'

dry_run_summary = parse_stdout_json(str(dry_run_result['stdout']))
assert 'protocol_output_dir' in dry_run_summary, 'Dry-run summary missing protocol_output_dir.'

dry_protocol_output_dir = Path(str(dry_run_summary['protocol_output_dir']))
assert dry_protocol_output_dir.exists(), f'Dry-run protocol output dir missing: {dry_protocol_output_dir}'

ensure_files_exist(dry_protocol_output_dir, EXPECTED_PROTOCOL_FILES, label='Dry-run protocol artifacts')

dry_execution = load_json(dry_protocol_output_dir / 'execution_status.json')
assert dry_execution.get('dry_run') is True, 'Expected execution_status.json dry_run=true for dry-run.'
assert dry_execution.get('runs'), 'Dry-run execution_status.json contains no runs.'
assert all(run.get('status') == 'planned' for run in dry_execution['runs']), 'Expected all dry-run statuses to be planned.'

print('Dry-run protocol output directory tree:')
print_tree(dry_protocol_output_dir, max_depth=2)

dry_report_index = pd.read_csv(dry_protocol_output_dir / 'report_index.csv')
assert not dry_report_index.empty, 'Dry-run report_index.csv is empty.'

suite_counts = dry_report_index.groupby('suite_id')['run_id'].count().sort_values()
print('Planned runs per suite (from dry-run report_index.csv):')
print(suite_counts.to_string())

SMALLEST_SUITE_ID = str(suite_counts.index[0])
print('Selected smallest suite for real execution:', SMALLEST_SUITE_ID)


$ 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe' -m Thesis_ML.cli.protocol_runner --protocol 'D:\Khaled\Projects\VScode\Thesis\configs\protocols\thesis_canonical_v1.json' --all-suites --reports-root 'D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\canonical_protocol\dry_run_20260315_020835' --dry-run
Return code: 0
----- STDOUT -----
{
  "protocol_id": "thesis-canonical",
  "protocol_version": "1.0.0",
  "protocol_output_dir": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\canonical_protocol\\dry_run_20260315_020835\\protocol_runs\\thesis-canonical__1.0.0",
  "n_completed": 0,
  "n_failed": 0,
  "n_planned": 8,
  "artifact_paths": {
    "protocol_json": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\canonical_protocol\\dry_run_20260315_020835\\protocol_runs\\thesis-canonical__1.0.0\\protocol.json",
    "compiled_protocol_manifest": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\canonical_protocol\\dry_r

The smallest suite is selected from real compiled run counts in dry-run output (`report_index.csv`), not guessed manually.

## Real Execution for One Small Official Suite

In [6]:
real_run_cmd = CLI_BASE_CMD + [
    '--protocol', str(PROTOCOL_PATH),
    '--suite', SMALLEST_SUITE_ID,
    '--reports-root', str(REAL_RUN_REPORTS_ROOT),
]

real_run_result = run_subprocess(real_run_cmd)
assert real_run_result['returncode'] == 0, 'Real suite run command failed.'

real_run_summary = parse_stdout_json(str(real_run_result['stdout']))
assert int(real_run_summary.get('n_failed', -1)) == 0, f'Real run reported failures: {real_run_summary}'
assert int(real_run_summary.get('n_completed', 0)) > 0, 'Real run completed zero runs.'

real_protocol_output_dir = Path(str(real_run_summary['protocol_output_dir']))
assert real_protocol_output_dir.exists(), f'Real run protocol output dir missing: {real_protocol_output_dir}'

print('Real run summary:')
print(json.dumps(real_run_summary, indent=2))


$ 'd:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe' -m Thesis_ML.cli.protocol_runner --protocol 'D:\Khaled\Projects\VScode\Thesis\configs\protocols\thesis_canonical_v1.json' --suite primary_within_subject --reports-root 'D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\canonical_protocol\real_run_20260315_020835'
Return code: 0
----- STDOUT -----
{
  "protocol_id": "thesis-canonical",
  "protocol_version": "1.0.0",
  "protocol_output_dir": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\canonical_protocol\\real_run_20260315_020835\\protocol_runs\\thesis-canonical__1.0.0",
  "n_completed": 2,
  "n_failed": 0,
  "n_planned": 0,
  "artifact_paths": {
    "protocol_json": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\canonical_protocol\\real_run_20260315_020835\\protocol_runs\\thesis-canonical__1.0.0\\protocol.json",
    "compiled_protocol_manifest": "D:\\Khaled\\Projects\\VScode\\Thesis\\outputs\\manual_validation\\canonical_prot

## Artifact Inspection

In [7]:
ensure_files_exist(real_protocol_output_dir, EXPECTED_PROTOCOL_FILES, label='Real-run protocol artifacts')

print('Protocol-level artifact directory tree:')
print_tree(real_protocol_output_dir, max_depth=3)

report_index_path = real_protocol_output_dir / 'report_index.csv'
report_index = pd.read_csv(report_index_path)
assert not report_index.empty, 'Real-run report_index.csv is empty.'

print('report_index.csv preview:')
print(report_index.head(10).to_string(index=False))

completed_rows = report_index[report_index['status'] == 'completed'].copy()
assert not completed_rows.empty, 'No completed runs found in report_index.csv.'

sample_row = completed_rows.iloc[0]
SAMPLE_RUN_ID = str(sample_row['run_id'])
SAMPLE_REPORT_DIR = Path(str(sample_row['report_dir']))
assert SAMPLE_REPORT_DIR.exists(), f'Sample report_dir does not exist: {SAMPLE_REPORT_DIR}'

ensure_files_exist(SAMPLE_REPORT_DIR, EXPECTED_RUN_FILES, label=f'Run-level artifacts for {SAMPLE_RUN_ID}')

print('Sample run_id:', SAMPLE_RUN_ID)
print('Sample report_dir:', SAMPLE_REPORT_DIR)
print('Sample run artifact tree:')
print_tree(SAMPLE_REPORT_DIR, max_depth=2)


Protocol-level artifact directory tree:
D:\Khaled\Projects\VScode\Thesis\outputs\manual_validation\canonical_protocol\real_run_20260315_020835\protocol_runs\thesis-canonical__1.0.0
  claim_to_run_map.json
  compiled_protocol_manifest.json
  execution_status.json
  protocol.json
  report_index.csv
  suite_summary.json
report_index.csv preview:
                                                               run_id               suite_id                                 claim_ids    status        target model                     cv_mode subject  train_subject  test_subject  seed    primary_metric  permutation_enabled  n_permutations  permutation_metric  dummy_baseline_run  interpretability_enabled  canonical_run                                                                                                                                                                   report_dir                                                                                                               

## Protocol Metadata Validation

In [8]:
config_payload = load_json(SAMPLE_REPORT_DIR / 'config.json')
metrics_payload = load_json(SAMPLE_REPORT_DIR / 'metrics.json')

required_metadata_keys = [
    'canonical_run',
    'protocol_id',
    'protocol_version',
    'protocol_schema_version',
    'suite_id',
    'claim_ids',
]

for payload_name, payload in (('config.json', config_payload), ('metrics.json', metrics_payload)):
    missing_keys = [key for key in required_metadata_keys if key not in payload]
    assert not missing_keys, f'{payload_name} is missing protocol metadata keys: {missing_keys}'

assert config_payload['canonical_run'] is True, 'config.json canonical_run should be true for protocol runs.'
assert metrics_payload['canonical_run'] is True, 'metrics.json canonical_run should be true for protocol runs.'
assert config_payload['suite_id'] == SMALLEST_SUITE_ID, 'config.json suite_id mismatch.'
assert metrics_payload['suite_id'] == SMALLEST_SUITE_ID, 'metrics.json suite_id mismatch.'
assert isinstance(config_payload['claim_ids'], list) and config_payload['claim_ids'], 'config.json claim_ids must be a non-empty list.'
assert isinstance(metrics_payload['claim_ids'], list) and metrics_payload['claim_ids'], 'metrics.json claim_ids must be a non-empty list.'

print('Protocol metadata keys are present in config.json and metrics.json.')
print('config.json metadata snippet:')
print_json_snippet(SAMPLE_REPORT_DIR / 'config.json', max_chars=1400)
print('metrics.json metadata snippet:')
print_json_snippet(SAMPLE_REPORT_DIR / 'metrics.json', max_chars=1400)


Protocol metadata keys are present in config.json and metrics.json.
config.json metadata snippet:
{
  "run_id": "canonical_thesis_canonical_1_0_0_primary_within_subject_ridge_sub_001",
  "timestamp": "20260315_020902",
  "index_csv": "D:\\Khaled\\Projects\\VScode\\Thesis\\Data\\processed\\dataset_index.csv",
  "data_root": "D:\\Khaled\\Projects\\VScode\\Thesis\\Data",
  "cache_dir": "D:\\Khaled\\Projects\\VScode\\Thesis\\Data\\processed\\feature_cache",
  "target": "coarse_affect",
  "model": "ridge",
  "cv": "within_subject_loso_session",
  "experiment_mode": "within_subject_loso_session",
  "subject": "sub-001",
  "train_subject": null,
  "test_subject": null,
  "seed": 42,
  "primary_metric_name": "balanced_accuracy",
  "permutation_metric_name": "balanced_accuracy",
  "filter_task": null,
  "filter_modality": null,
  "n_permutations": 0,
  "canonical_run": true,
  "protocol_id": "thesis-canonical",
  "protocol_version": "1.0.0",
  "protocol_schema_version": "thesis-protocol-v1",
  

## Final Verdict

In [9]:
final_checks = {
    'dry_run_command_succeeded': dry_run_result['returncode'] == 0,
    'dry_run_protocol_artifacts_present': all((dry_protocol_output_dir / name).exists() for name in EXPECTED_PROTOCOL_FILES),
    'real_run_command_succeeded': real_run_result['returncode'] == 0,
    'real_run_had_no_failures': int(real_run_summary.get('n_failed', -1)) == 0,
    'real_run_protocol_artifacts_present': all((real_protocol_output_dir / name).exists() for name in EXPECTED_PROTOCOL_FILES),
    'sample_run_artifacts_present': all((SAMPLE_REPORT_DIR / name).exists() for name in EXPECTED_RUN_FILES),
    'protocol_metadata_in_config': all(key in config_payload for key in required_metadata_keys),
    'protocol_metadata_in_metrics': all(key in metrics_payload for key in required_metadata_keys),
}

for check_name, passed in final_checks.items():
    print(f'{check_name}: {'PASS' if passed else 'FAIL'}')

failed_checks = [name for name, passed in final_checks.items() if not passed]
assert not failed_checks, f'Manual validation FAILED. Failed checks: {failed_checks}'
print('\nManual validation PASS: canonical protocol dry-run + real-suite + artifact inspection completed successfully.')


dry_run_command_succeeded: PASS
dry_run_protocol_artifacts_present: PASS
real_run_command_succeeded: PASS
real_run_had_no_failures: PASS
real_run_protocol_artifacts_present: PASS
sample_run_artifacts_present: PASS
protocol_metadata_in_config: PASS
protocol_metadata_in_metrics: PASS

Manual validation PASS: canonical protocol dry-run + real-suite + artifact inspection completed successfully.
